# Thai-Depression — Frozen WangchanBERTa + Optuna + 5 Independent Runs

ไฟล์นี้ยึดจาก notebook ที่รันจริง:
`wanchanberta-x-transformers-thai-depression-optim(6).ipynb`

## Protocol
- Dataset: official `train.json`, `valid.json`, `test.json`
- Labels: `no_depression`, `depression`
- Encoder: `airesearch/wangchanberta-base-att-spm-uncased`
- **Frozen encoder**
- X-Transformers decoder + Attention Pooling + classifier
- CrossEntropyLoss + class weights เหมือน notebook เดิม
- Batch size = 64
- Max length = 128
- Final training = 100 epochs
- Optuna ทำ **ครั้งเดียว**
- Best hyperparameters ชุดเดียวกันใช้กับ final training ทั้ง 5 seeds
- Official train / validation / test ไม่เปลี่ยน
- สรุป Mean / Median / Sample Variance / Sample SD

## หมายเหตุสำคัญ
ผลเดิม `Accuracy=0.8183, Macro-F1=0.8182` มาจากโมเดลที่ **ไม่ได้ freeze encoder**
จึงไม่ถูกนำมารวมกับ 5 runs ใหม่นี้ เพราะจะเป็นคนละ experimental setup

เพื่อประหยัดเวลา frozen encoder จะ encode ข้อมูลเพียงครั้งเดียว แล้ว Optuna / 5 final runs
จะ train เฉพาะ decoder + pooling + classifier

In [ ]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn

In [ ]:
import os
import glob
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

import optuna

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)

In [ ]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
EPOCHS = 100
BATCH_SIZE = 64

NUM_LABELS = 2

LABEL_NAMES = [
    "no_depression",
    "depression"
]

# Optuna ทำครั้งเดียว
N_TRIALS = 10
OPTUNA_EPOCHS = 35

# 5 final independent training seeds
TRAIN_SEEDS = [
    13,
    21,
    42,
    87,
    123
]

OPTUNA_SEED = 42

# best params เดิมจาก notebook ที่ไม่ freeze
# จะถูก enqueue เป็น trial แรก เพื่อไม่ทิ้งข้อมูลจากการค้นหาเดิม
OLD_BEST_PARAMS = {
    "depth": 2,
    "heads": 4,
    "attn_dropout": 0.2240577833126539,
    "ff_dropout": 0.4922163075549199,
    "ff_mult": 4,
    "lr": 3.0331847361552553e-05,
    "weight_decay": 0.0002821544949248052
}

OUTPUT_DIR = (
    "/kaggle/working"
    if os.path.exists("/kaggle/working")
    else "/content"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# =========================================================
# REPRODUCIBILITY
# =========================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(
    OPTUNA_SEED
)

In [ ]:
# =========================================================
# FIND OFFICIAL THAI-DEPRESSION SPLITS
# =========================================================

def find_json(filename):

    candidates = [
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    hits = glob.glob(
        f"/kaggle/input/**/{filename}",
        recursive=True
    )

    # ถ้ามีหลาย dataset ให้เลือก path ที่มีคำว่า depression ก่อน
    depression_hits = [
        p for p in hits
        if "depression" in p.lower()
    ]

    if depression_hits:
        return depression_hits[0]

    if hits:
        return hits[0]

    raise FileNotFoundError(
        f"ไม่พบ {filename} กรุณา Add Thai Depression dataset "
        f"ใน Kaggle หรือ upload เข้า /content"
    )


TRAIN_PATH = find_json(
    "train.json"
)

VALID_PATH = find_json(
    "valid.json"
)

TEST_PATH = find_json(
    "test.json"
)

print("TRAIN:", TRAIN_PATH)
print("VALID:", VALID_PATH)
print("TEST :", TEST_PATH)

In [ ]:
# =========================================================
# LOAD DATA — preprocessing เหมือน notebook เดิม
# =========================================================

def load_json_dataset(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        raw_data = json.load(f)

    texts = []
    labels = []

    for item in raw_data:

        if not isinstance(
            item,
            (list, tuple)
        ):
            continue

        if len(item) != 2:
            continue

        text, label = item

        if not isinstance(
            text,
            str
        ):
            continue

        text = text.replace(
            "---- “ ----",
            " "
        )

        text = text.replace(
            "\n",
            " "
        )

        text = " ".join(
            text.split()
        )

        text = text.strip()

        if len(text) < 2:
            continue

        if label == "depression":
            label_id = 1

        elif label == "no_depression":
            label_id = 0

        else:
            continue

        texts.append(
            text
        )

        labels.append(
            label_id
        )

    return pd.DataFrame({
        "text": texts,
        "label": labels
    })


train_df = load_json_dataset(
    TRAIN_PATH
)

val_df = load_json_dataset(
    VALID_PATH
)

test_df = load_json_dataset(
    TEST_PATH
)

print("TRAIN:", len(train_df))
print("VALID:", len(val_df))
print("TEST :", len(test_df))

print("\nTRAIN LABEL COUNTS")
print(
    train_df["label"]
    .value_counts()
    .sort_index()
)

print("\nVALID LABEL COUNTS")
print(
    val_df["label"]
    .value_counts()
    .sort_index()
)

print("\nTEST LABEL COUNTS")
print(
    test_df["label"]
    .value_counts()
    .sort_index()
)

In [ ]:
# =========================================================
# CLASS WEIGHTS — สูตรเดิม
# =========================================================

class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .values
)

class_weights = (
    1.0
    / class_counts
)

class_weights = (
    class_weights
    / class_weights.sum()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(DEVICE)

print(
    "CLASS WEIGHTS:",
    class_weights
)

In [ ]:
# =========================================================
# TOKENIZER + FROZEN WANGCHANBERTA
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME
).to(DEVICE)

for param in encoder.parameters():
    param.requires_grad = False

encoder.eval()

HIDDEN_SIZE = (
    encoder.config.hidden_size
)

print(
    "HIDDEN SIZE:",
    HIDDEN_SIZE
)

print(
    "TRAINABLE ENCODER PARAMETERS:",
    sum(
        p.numel()
        for p in encoder.parameters()
        if p.requires_grad
    )
)

In [ ]:
# =========================================================
# CACHE FROZEN ENCODER OUTPUTS ONCE
# =========================================================

class RawTextDataset(Dataset):

    def __init__(self, frame):

        self.texts = (
            frame["text"]
            .tolist()
        )

        self.labels = (
            frame["label"]
            .values
            .astype(np.int64)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(
        self,
        idx
    ):

        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids":
                enc["input_ids"]
                .squeeze(0),

            "attention_mask":
                enc["attention_mask"]
                .squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }


@torch.no_grad()
def precompute_embeddings(
    frame,
    split_name
):

    loader = DataLoader(
        RawTextDataset(frame),
        batch_size=32,
        shuffle=False
    )

    all_hidden = []
    all_masks = []
    all_labels = []

    use_amp = (
        DEVICE.type == "cuda"
    )

    for batch in tqdm(
        loader,
        desc=f"Encoding {split_name}"
    ):

        input_ids = (
            batch["input_ids"]
            .to(DEVICE)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(DEVICE)
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            hidden = encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).last_hidden_state

        all_hidden.append(
            hidden
            .half()
            .cpu()
        )

        all_masks.append(
            attention_mask
            .bool()
            .cpu()
        )

        all_labels.append(
            batch["labels"]
            .cpu()
        )

    return (
        torch.cat(
            all_hidden,
            dim=0
        ),
        torch.cat(
            all_masks,
            dim=0
        ),
        torch.cat(
            all_labels,
            dim=0
        )
    )


print("Encoding TRAIN...")
train_hidden, train_masks, train_labels = (
    precompute_embeddings(
        train_df,
        "train"
    )
)

print("Encoding VALID...")
val_hidden, val_masks, val_labels = (
    precompute_embeddings(
        val_df,
        "validation"
    )
)

print("Encoding TEST...")
test_hidden, test_masks, test_labels = (
    precompute_embeddings(
        test_df,
        "test"
    )
)

print(
    "TRAIN HIDDEN:",
    train_hidden.shape,
    train_hidden.dtype
)

print(
    "VALID HIDDEN:",
    val_hidden.shape,
    val_hidden.dtype
)

print(
    "TEST HIDDEN:",
    test_hidden.shape,
    test_hidden.dtype
)

del encoder

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# =========================================================
# CACHED DATASET
# =========================================================

class CachedDataset(Dataset):

    def __init__(
        self,
        hidden,
        masks,
        labels
    ):

        self.hidden = hidden
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(
            self.hidden
        )

    def __getitem__(
        self,
        idx
    ):

        return {
            "hidden":
                self.hidden[idx],

            "attention_mask":
                self.masks[idx],

            "labels":
                self.labels[idx]
        }


train_dataset = CachedDataset(
    train_hidden,
    train_masks,
    train_labels
)

val_dataset = CachedDataset(
    val_hidden,
    val_masks,
    val_labels
)

test_dataset = CachedDataset(
    test_hidden,
    test_masks,
    test_labels
)


def make_loaders(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )

In [ ]:
# =========================================================
# MODEL
# =========================================================

class AttentionPooling(nn.Module):

    def __init__(
        self,
        hidden_size
    ):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(
                hidden_size,
                hidden_size
            ),
            nn.Tanh(),
            nn.Linear(
                hidden_size,
                1
            )
        )

    def forward(
        self,
        x,
        mask
    ):

        scores = (
            self.attention(x)
            .squeeze(-1)
        )

        scores = scores.masked_fill(
            ~mask.bool(),
            torch.finfo(
                scores.dtype
            ).min
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x
            * weights.unsqueeze(-1),
            dim=1
        )

        return pooled


class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.decoder = Decoder(
            dim=HIDDEN_SIZE,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(
            HIDDEN_SIZE
        )

        self.dropout = nn.Dropout(
            0.4
        )

        self.fc = nn.Linear(
            HIDDEN_SIZE,
            NUM_LABELS
        )

    def forward(
        self,
        hidden,
        attention_mask
    ):

        dtype = next(
            self.parameters()
        ).dtype

        x = hidden.to(
            DEVICE,
            dtype=dtype
        )

        mask = attention_mask.to(
            DEVICE
        )

        # คงโครง decoder จาก notebook เดิม
        x = self.decoder(
            x
        )

        x = self.pooling(
            x,
            mask
        )

        x = self.dropout(
            x
        )

        return self.fc(
            x
        )

In [ ]:
# =========================================================
# EVALUATION
# =========================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    all_labels = []
    all_preds = []

    for batch in loader:

        logits = model(
            batch["hidden"],
            batch["attention_mask"]
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            batch["labels"]
            .cpu()
            .numpy()
        )

        all_preds.extend(
            preds
            .cpu()
            .numpy()
        )

    y_true = np.array(
        all_labels
    )

    y_pred = np.array(
        all_preds
    )

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "micro_precision":
            precision_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "micro_recall":
            recall_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "micro_f1":
            f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "per_label_precision":
            precision_score(
                y_true,
                y_pred,
                average=None,
                labels=[0, 1],
                zero_division=0
            ),

        "per_label_recall":
            recall_score(
                y_true,
                y_pred,
                average=None,
                labels=[0, 1],
                zero_division=0
            ),

        "per_label_f1":
            f1_score(
                y_true,
                y_pred,
                average=None,
                labels=[0, 1],
                zero_division=0
            ),

        "labels":
            y_true,

        "preds":
            y_pred
    }

In [ ]:
# =========================================================
# TRAIN DECODER / POOLING / CLASSIFIER ONLY
# =========================================================

def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    epochs,
    verbose=False
):

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = (
        len(train_loader)
        * epochs
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(
                0.1 * total_steps
            ),
            num_training_steps=total_steps
        )
    )

    use_amp = (
        DEVICE.type == "cuda"
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(
        epochs
    ):

        model.train()
        total_loss = 0.0

        for batch in train_loader:

            labels = (
                batch["labels"]
                .to(DEVICE)
            )

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    batch["hidden"],
                    batch["attention_mask"]
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            total_loss += (
                loss.item()
            )

        if verbose and (
            epoch == 0
            or (epoch + 1) % 10 == 0
            or epoch == epochs - 1
        ):

            val_result = evaluate(
                model,
                val_loader
            )

            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"loss={total_loss:.4f} | "
                f"val_acc={val_result['accuracy']:.4f} | "
                f"val_micro={val_result['micro_f1']:.4f} | "
                f"val_macro={val_result['macro_f1']:.4f}"
            )

    return model

In [ ]:
# =========================================================
# OPTUNA — RUN ONCE ON FROZEN SETUP
# =========================================================

optuna_train_loader, (
    optuna_val_loader
), _ = make_loaders(
    OPTUNA_SEED
)


def objective(trial):

    seed_everything(
        OPTUNA_SEED
    )

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(DEVICE)

    model = train_model(
        model,
        optuna_train_loader,
        optuna_val_loader,
        lr=lr,
        weight_decay=weight_decay,
        epochs=OPTUNA_EPOCHS,
        verbose=False
    )

    val_result = evaluate(
        model,
        optuna_val_loader
    )

    score = (
        val_result["macro_f1"]
    )

    trial.set_user_attr(
        "val_accuracy",
        float(
            val_result["accuracy"]
        )
    )

    trial.set_user_attr(
        "val_micro_f1",
        float(
            val_result["micro_f1"]
        )
    )

    trial.set_user_attr(
        "val_macro_f1",
        float(
            val_result["macro_f1"]
        )
    )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return score


sampler = optuna.samplers.TPESampler(
    seed=OPTUNA_SEED
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# ลอง best params เดิมเป็น trial แรก
study.enqueue_trial(
    OLD_BEST_PARAMS
)

study.optimize(
    objective,
    n_trials=N_TRIALS
)

best_params = (
    study.best_params
)

print("\nBEST OPTUNA SCORE:")
print(
    study.best_value
)

print("\nBEST PARAMS:")
print(
    best_params
)

best_params_path = os.path.join(
    OUTPUT_DIR,
    "thai_depression_frozen_optuna_best_params.json"
)

with open(
    best_params_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_params,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Saved:",
    best_params_path
)

In [ ]:
# =========================================================
# FINAL 5 INDEPENDENT RUNS
# =========================================================

all_run_rows = []
all_per_label_rows = []

for run_idx, seed in enumerate(
    TRAIN_SEEDS,
    start=1
):

    print("\n" + "=" * 90)
    print(
        f"FINAL RUN {run_idx}/5 | seed={seed}"
    )
    print("=" * 90)

    seed_everything(
        seed
    )

    (
        train_loader,
        val_loader,
        test_loader
    ) = make_loaders(
        seed
    )

    model = EmotionModel(
        depth=best_params["depth"],
        heads=best_params["heads"],
        attn_dropout=best_params["attn_dropout"],
        ff_dropout=best_params["ff_dropout"],
        ff_mult=best_params["ff_mult"]
    ).to(DEVICE)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        epochs=EPOCHS,
        verbose=True
    )

    # ใช้ final epoch model ตาม protocol เดียวกันทุก run
    test_result = evaluate(
        model,
        test_loader
    )

    row = {
        "run":
            run_idx,

        "seed":
            seed,

        "accuracy":
            test_result[
                "accuracy"
            ],

        "micro_precision":
            test_result[
                "micro_precision"
            ],

        "micro_recall":
            test_result[
                "micro_recall"
            ],

        "micro_f1":
            test_result[
                "micro_f1"
            ],

        "macro_precision":
            test_result[
                "macro_precision"
            ],

        "macro_recall":
            test_result[
                "macro_recall"
            ],

        "macro_f1":
            test_result[
                "macro_f1"
            ],

        "weighted_f1":
            test_result[
                "weighted_f1"
            ]
    }

    all_run_rows.append(
        row
    )

    print("\nTEST RESULT")

    for key, value in row.items():

        if key not in [
            "run",
            "seed"
        ]:

            print(
                f"{key:20s}: "
                f"{value:.6f}"
            )

    for i, label_name in enumerate(
        LABEL_NAMES
    ):

        all_per_label_rows.append({
            "run":
                run_idx,

            "seed":
                seed,

            "label":
                label_name,

            "precision":
                test_result[
                    "per_label_precision"
                ][i],

            "recall":
                test_result[
                    "per_label_recall"
                ][i],

            "f1":
                test_result[
                    "per_label_f1"
                ][i],

            "support":
                int(
                    np.sum(
                        test_result["labels"]
                        == i
                    )
                )
        })

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


results_df = pd.DataFrame(
    all_run_rows
)

per_label_df = pd.DataFrame(
    all_per_label_rows
)

print("\nALL 5 RUNS")
display(
    results_df
)

In [ ]:
# =========================================================
# MEAN / MEDIAN / VARIANCE / STANDARD DEVIATION
# =========================================================

metric_columns = [
    "accuracy",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1"
]

summary_rows = []

for metric in metric_columns:

    values = (
        results_df[metric]
        .astype(float)
    )

    summary_rows.append({
        "metric":
            metric,

        "mean":
            values.mean(),

        "median":
            values.median(),

        # sample variance
        "variance":
            values.var(
                ddof=1
            ),

        # sample standard deviation
        "std":
            values.std(
                ddof=1
            ),

        "min":
            values.min(),

        "max":
            values.max()
    })


summary_df = pd.DataFrame(
    summary_rows
)

print(
    "\n5-RUN STATISTICAL SUMMARY"
)

display(
    summary_df
)

In [ ]:
# =========================================================
# PER-LABEL SUMMARY
# =========================================================

per_label_summary = (
    per_label_df
    .groupby(
        "label",
        sort=False
    )
    .agg(
        precision_mean=(
            "precision",
            "mean"
        ),

        precision_median=(
            "precision",
            "median"
        ),

        precision_variance=(
            "precision",
            lambda x: x.var(
                ddof=1
            )
        ),

        precision_std=(
            "precision",
            lambda x: x.std(
                ddof=1
            )
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_median=(
            "recall",
            "median"
        ),

        recall_variance=(
            "recall",
            lambda x: x.var(
                ddof=1
            )
        ),

        recall_std=(
            "recall",
            lambda x: x.std(
                ddof=1
            )
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_median=(
            "f1",
            "median"
        ),

        f1_variance=(
            "f1",
            lambda x: x.var(
                ddof=1
            )
        ),

        f1_std=(
            "f1",
            lambda x: x.std(
                ddof=1
            )
        )
    )
    .reset_index()
)

print(
    "\nPER-LABEL 5-RUN SUMMARY"
)

display(
    per_label_summary
)

In [ ]:
# =========================================================
# EXPORT
# =========================================================

paths = {
    "raw_results":
        os.path.join(
            OUTPUT_DIR,
            "thai_depression_frozen_optuna_5runs_raw.csv"
        ),

    "summary":
        os.path.join(
            OUTPUT_DIR,
            "thai_depression_frozen_optuna_5runs_summary.csv"
        ),

    "per_label_raw":
        os.path.join(
            OUTPUT_DIR,
            "thai_depression_frozen_optuna_5runs_per_label_raw.csv"
        ),

    "per_label_summary":
        os.path.join(
            OUTPUT_DIR,
            "thai_depression_frozen_optuna_5runs_per_label_summary.csv"
        ),

    "optuna_trials":
        os.path.join(
            OUTPUT_DIR,
            "thai_depression_frozen_optuna_trials.csv"
        )
}


results_df.to_csv(
    paths["raw_results"],
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    paths["summary"],
    index=False,
    encoding="utf-8-sig"
)

per_label_df.to_csv(
    paths["per_label_raw"],
    index=False,
    encoding="utf-8-sig"
)

per_label_summary.to_csv(
    paths["per_label_summary"],
    index=False,
    encoding="utf-8-sig"
)

study.trials_dataframe().to_csv(
    paths["optuna_trials"],
    index=False,
    encoding="utf-8-sig"
)

print("EXPORTED FILES")

for name, path in paths.items():
    print(
        name,
        "->",
        path
    )